# Recover missing `.bbl` files

This notebook retries exact paper versions whose journal says a `.bbl` was dropped and whose retained files contain neither `.bib` nor `.bbl`. Files and logs are stored in a separate Google Drive folder; the original corpus is not changed.

**Run one Colab session at a time.** Input checks are offline and `RUN_NETWORK` defaults to `False`. The first run is limited to 10 versions; after reviewing those results, set `LIMIT = None` to continue with all candidates.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## Prepare the runtime

Upload `recovery-runtime.zip` from the BBL recovery kit to `MyDrive/bbl-recovery-kit/`. The runtime contains code and the five journals, not the original corpus. Grant Drive access when prompted.


In [ ]:
import hashlib
import json
import subprocess
import sys
import zipfile
from pathlib import Path

# Upload recovery-runtime.zip to this Drive folder.
KIT = Path("/content/drive/MyDrive/bbl-recovery-kit")
ARCHIVE = KIT / "recovery-runtime.zip"
assert ARCHIVE.is_file(), (
    f"Cannot find {ARCHIVE}. Upload recovery-runtime.zip to KIT or change the KIT path."
)
REPO = Path("/content/recovery-runtime")
REPO.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ARCHIVE) as bundle:
    for member in bundle.infolist():
        target = (REPO / member.filename).resolve()
        if not target.is_relative_to(REPO.resolve()):
            raise ValueError("Unsafe archive path")
    bundle.extractall(REPO)
manifest = json.loads((REPO / "runtime-manifest.json").read_text())
for name, expected in manifest["sha256"].items():
    with (REPO / name).open("rb") as stream:
        actual = hashlib.file_digest(stream, "sha256").hexdigest()
    assert actual == expected, f"Hash mismatch: {name}"
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements-recovery.txt")]
)
sys.path.insert(0, str(REPO))
print("Runtime and five journals are ready:", REPO)

In [ ]:
from src.benchmark.bbl_recovery import recover, select_candidates
from src.benchmark.package_recovery import RUNS

JOURNALS = {name: REPO / "evidence" / name / "work" / "versions.jsonl" for name in RUNS}
OUTPUT = KIT / "results" / "run-01"
RUN_NETWORK = False  # Set to True after checking the inputs.
LIMIT = 10  # Use None after reviewing the initial run.

candidates = select_candidates(JOURNALS)
print("Journals:", len(JOURNALS))
print("Candidate papers:", len({r["arxiv_id"] for r in candidates}))
print("Candidate versions:", len(candidates))
print("Output:", OUTPUT)
print("Limit:", LIMIT)
print("Network enabled:", RUN_NETWORK)

## Run recovery

After preflight, change `RUN_NETWORK` to `True` in the preceding cell and rerun it. Then run the next cell. Requests use one connection with at least a three-second interval. The full run needs at least about 115 minutes for 2,297 requests, plus download and retry time; Colab can disconnect sooner.


In [ ]:
if RUN_NETWORK:
    print("Version status counts:", recover(JOURNALS, OUTPUT, limit=LIMIT))
else:
    print("Preflight only; no downloads were requested.")

## Results and resume

`results/run-01/files/` contains recovered `.bbl` files under paper/version folders; `versions.jsonl` records each attempt and hashes, `summary.json` provides counts, and `candidates.json` and `manifest.json` preserve the inputs. `recovered` means at least one `.bbl` was saved; `no_bbl` means a complete readable archive contained none; `unavailable` means HTTP 404/410; `inconclusive` and `unsafe_or_limit` need review.

To resume, run the cells again with the same runtime ZIP and output folder. Recovered files are skipped only if their hashes match; unresolved cases are retried. `unavailable` and `no_bbl` are final for that run; use a new output folder for an independent retry. If a runtime is lost, `.running` may remain: check that no other session is writing before manually removing that folder. If the journal is damaged, preserve it for inspection before resuming.

The S3 v1 label is inferred from a snapshot; a source fetched from arXiv v1 is not proof of byte identity with the S3 archive. This candidate set is targeted and is not an estimate for every paper missing `.bib`. Review the outputs before any corpus integration.
